# Credit Scoring Engine - Complete Example

This notebook demonstrates the complete workflow of building a credit scorecard using the credit-scoring-engine library.

## Workflow:
1. Data preparation and exploration
2. Variable binning (continuous, categorical, ordinal)
3. Weight of Evidence (WoE) transformation
4. Information Value (IV) calculation
5. Model training (Logistic Regression)
6. Scorecard creation
7. Model evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Import credit scoring modules
from credit_scoring.binning import ContinuousBinning, CategoricalBinning, BinningEvaluator
from credit_scoring.woe import WoEEncoder, InformationValue, DScore
from credit_scoring.models import LogisticRegressionScorecard, ScorecardModel
from credit_scoring.metrics import (
    gini_coefficient,
    ks_statistic,
    population_stability_index,
    hosmer_lemeshow_test
)

## 1. Generate Sample Data

For demonstration purposes, we'll create synthetic credit data.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate sample data
n_samples = 10000

# Create features
age = np.random.normal(40, 12, n_samples)
income = np.random.lognormal(10, 0.8, n_samples)
debt_ratio = np.random.beta(2, 5, n_samples)
credit_history = np.random.choice(['excellent', 'good', 'fair', 'poor'], n_samples, p=[0.2, 0.3, 0.3, 0.2])
employment_type = np.random.choice(['permanent', 'contract', 'self-employed', 'unemployed'], n_samples, p=[0.5, 0.3, 0.15, 0.05])

# Generate target with some correlation to features
risk_score = (
    -0.02 * age +
    -0.00001 * income +
    3 * debt_ratio +
    np.random.normal(0, 1, n_samples)
)

# Add credit history effect
history_effect = {'excellent': -1.5, 'good': -0.5, 'fair': 0.5, 'poor': 1.5}
risk_score += np.array([history_effect[h] for h in credit_history])

# Convert to probability
prob = 1 / (1 + np.exp(-risk_score))
default = (np.random.random(n_samples) < prob).astype(int)

# Create DataFrame
df = pd.DataFrame({
    'age': age,
    'income': income,
    'debt_ratio': debt_ratio,
    'credit_history': credit_history,
    'employment_type': employment_type,
    'default': default
})

print(f"Dataset shape: {df.shape}")
print(f"Default rate: {df['default'].mean():.2%}")
df.head()

## 2. Train-Test Split

In [ ]:
# Split data
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['default'])

print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")
print(f"Train default rate: {train_df['default'].mean():.2%}")
print(f"Test default rate: {test_df['default'].mean():.2%}")

## 3. Binning - Continuous Variables

We'll use different binning methods for continuous variables.

In [ ]:
# Age - Quantile binning
age_binner = ContinuousBinning(method='quantile', n_bins=5)
train_df['age_binned'] = age_binner.fit_transform(train_df['age'], train_df['default'])
test_df['age_binned'] = age_binner.transform(test_df['age'])

# Income - Tree-based binning
income_binner = ContinuousBinning(method='tree', max_depth=3, min_samples_leaf=50)
train_df['income_binned'] = income_binner.fit_transform(train_df['income'], train_df['default'])
test_df['income_binned'] = income_binner.transform(test_df['income'])

# Debt ratio - Quantile binning
debt_binner = ContinuousBinning(method='quantile', n_bins=5)
train_df['debt_ratio_binned'] = debt_binner.fit_transform(train_df['debt_ratio'], train_df['default'])
test_df['debt_ratio_binned'] = debt_binner.transform(test_df['debt_ratio'])

print("\n=== Age Binning Statistics ===")
print(age_binner.get_bin_statistics(train_df['age'], train_df['default']))

print("\n=== Income Binning Statistics ===")
print(income_binner.get_bin_statistics(train_df['income'], train_df['default']))

## 4. Evaluate Binning Quality

In [ ]:
# Evaluate age binning
evaluator = BinningEvaluator()
age_quality = evaluator.evaluate_binning_quality(train_df['age_binned'], train_df['default'])

print("\n=== Age Binning Quality ===")
print(age_quality)

## 5. Binning - Categorical Variables

In [ ]:
# Credit history - Keep all categories
credit_binner = CategoricalBinning(method='keep_all')
train_df['credit_history_binned'] = credit_binner.fit_transform(train_df['credit_history'], train_df['default'])
test_df['credit_history_binned'] = credit_binner.transform(test_df['credit_history'])

# Employment type - Group rare categories
employment_binner = CategoricalBinning(method='rare', rare_threshold=0.1)
train_df['employment_type_binned'] = employment_binner.fit_transform(train_df['employment_type'], train_df['default'])
test_df['employment_type_binned'] = employment_binner.transform(test_df['employment_type'])

print("\n=== Credit History Statistics ===")
print(credit_binner.get_bin_statistics(train_df['credit_history'], train_df['default']))

## 6. Weight of Evidence (WoE) Transformation

In [ ]:
# Create WoE encoders for each feature
woe_encoders = {}
binned_features = ['age_binned', 'income_binned', 'debt_ratio_binned', 'credit_history_binned', 'employment_type_binned']

# Train and transform
train_woe = pd.DataFrame()
test_woe = pd.DataFrame()

for feature in binned_features:
    woe_encoder = WoEEncoder(smooth=0.5)
    train_woe[feature] = woe_encoder.fit_transform(train_df[feature], train_df['default'])
    test_woe[feature] = woe_encoder.transform(test_df[feature])
    woe_encoders[feature] = woe_encoder
    
    print(f"\n=== {feature} WoE Statistics ===")
    print(woe_encoder.get_woe_stats())

## 7. Information Value (IV) Calculation

In [ ]:
# Calculate IV for all features
iv_calc = InformationValue()
iv_summary = iv_calc.calculate_multiple(train_df[binned_features], train_df['default'])

print("\n=== Information Value Summary ===")
print(iv_summary)

## 8. d-score Calculation

In [ ]:
# Calculate d-score for WoE-transformed features
dscore_summary = DScore.calculate_multiple(train_woe, train_df['default'])

print("\n=== D-Score Summary ===")
print(dscore_summary)

## 9. Train Logistic Regression Model

In [ ]:
# Train model with L2 regularization
model = LogisticRegressionScorecard(penalty='l2', C=1.0, random_state=42)
model.fit(train_woe, train_df['default'], feature_names=binned_features)

# Get coefficients
coefficients = model.get_coefficients()
print("\n=== Model Coefficients ===")
print(coefficients)

# Get odds ratios
odds_ratios = model.get_odds_ratio()
print("\n=== Odds Ratios ===")
print(odds_ratios)

## 10. Model Predictions and Evaluation

In [ ]:
# Make predictions
train_pred_proba = model.predict_proba(train_woe)[:, 1]
test_pred_proba = model.predict_proba(test_woe)[:, 1]

# Calculate performance metrics
print("\n=== Training Set Performance ===")
train_gini = gini_coefficient(train_df['default'], train_pred_proba)
train_ks, train_ks_threshold = ks_statistic(train_df['default'], train_pred_proba)
print(f"Gini: {train_gini:.4f}")
print(f"KS: {train_ks:.4f} (at threshold {train_ks_threshold:.4f})")

print("\n=== Test Set Performance ===")
test_gini = gini_coefficient(test_df['default'], test_pred_proba)
test_ks, test_ks_threshold = ks_statistic(test_df['default'], test_pred_proba)
print(f"Gini: {test_gini:.4f}")
print(f"KS: {test_ks:.4f} (at threshold {test_ks_threshold:.4f})")

## 11. Population Stability Index (PSI)

In [ ]:
# Calculate PSI between train and test
psi = population_stability_index(train_pred_proba, test_pred_proba, bins=10)

print(f"\n=== Population Stability Index ===")
print(f"PSI: {psi:.4f}")

if psi < 0.1:
    print("✓ No significant population shift")
elif psi < 0.2:
    print("⚠ Moderate population shift - investigate further")
else:
    print("✗ Significant population shift - model may need recalibration")

## 12. Calibration Test

In [ ]:
# Hosmer-Lemeshow test
chi2, p_value = hosmer_lemeshow_test(test_df['default'], test_pred_proba, n_bins=10)

print(f"\n=== Hosmer-Lemeshow Test ===")
print(f"Chi-square: {chi2:.2f}")
print(f"P-value: {p_value:.4f}")

if p_value > 0.05:
    print("✓ Model is well calibrated (p > 0.05)")
else:
    print("⚠ Model may be poorly calibrated (p < 0.05)")

## 13. Create Scorecard

In [ ]:
# Create scorecard with base points = 600, PDO = 20
scorecard = ScorecardModel(base_points=600, pdo=20, base_odds=50)
scorecard.fit(coefficients, model.get_intercept())

# Get WoE values for scorecard
woe_values = {}
for feature, encoder in woe_encoders.items():
    woe_values[feature] = encoder.get_woe_mapping()

# Calculate points
scorecard_points = scorecard.calculate_points(woe_values)

print("\n=== Scorecard Points ===")
print(scorecard_points.head(20))

## 14. Score Distribution

In [ ]:
# Calculate scores for test set
test_scores = scorecard.transform(test_df[binned_features], woe_encoders)

print(f"\n=== Score Statistics ===")
print(f"Mean score: {np.mean(test_scores):.2f}")
print(f"Median score: {np.median(test_scores):.2f}")
print(f"Std score: {np.std(test_scores):.2f}")
print(f"Min score: {np.min(test_scores):.2f}")
print(f"Max score: {np.max(test_scores):.2f}")

# Plot score distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(test_scores[test_df['default'] == 0], bins=30, alpha=0.5, label='Non-default', density=True)
plt.hist(test_scores[test_df['default'] == 1], bins=30, alpha=0.5, label='Default', density=True)
plt.xlabel('Credit Score')
plt.ylabel('Density')
plt.title('Score Distribution by Default Status')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot([test_scores[test_df['default'] == 0], test_scores[test_df['default'] == 1]], 
            labels=['Non-default', 'Default'])
plt.ylabel('Credit Score')
plt.title('Score Distribution Box Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the complete credit scoring workflow:

1. ✓ Data preparation and train-test split
2. ✓ Variable binning with different methods
3. ✓ Binning quality evaluation
4. ✓ WoE transformation
5. ✓ Information Value calculation
6. ✓ d-score calculation
7. ✓ Logistic regression model training
8. ✓ Model evaluation (Gini, KS, PSI, Hosmer-Lemeshow)
9. ✓ Scorecard creation and points calculation
10. ✓ Score distribution analysis

The credit-scoring-engine library provides a comprehensive toolkit for building production-ready credit scoring models!